# 04 · EDA, preprocessing, and data leakage

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Measure missingness, split before fitting transforms, encode categories, and package preprocessing.


## Step 1 — Inspect a small mixed dataset

Real features can be numeric or categorical. Here both are NumPy arrays so pandas is unnecessary.


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
X = np.array([[20,'web'],[50,'store'],[None,'web'],[25,'app'],[55,'store'],[60,'app'],[22,'web'],[48,'store'],[30,'app'],[58,'web'],[23,'app'],[52,'store']], dtype=object)
y = np.array([0,1,0,0,1,1,0,1,0,1,0,1])
print('Rows:', len(X), 'missing ages:', sum(v is None for v in X[:,0]))
Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=42, test_size=0.33)


**Check your result:** There is one missing numeric value.


## Step 2 — Fit transformations inside a pipeline

Median and scaling statistics use training rows only. Unknown categories are handled safely at inference.


In [ ]:
numeric = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
preprocess = ColumnTransformer([
    ('numeric', numeric, [0]),
    ('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [1])])
model = make_pipeline(preprocess, LogisticRegression())
model.fit(Xtr, ytr)
print('Held-out predictions:', model.predict(Xte))


**Check your result:** The complete preprocessing and estimator run as one object.


## Step 3 — Check an unseen category

Robust preprocessing should handle valid but previously unseen category values.


In [ ]:
prediction = model.predict(np.array([[35, 'partner']], dtype=object))
assert prediction.shape == (1,)
print('Unseen category prediction:', prediction[0])


**Check your result:** Scoring succeeds even though partner was absent during training.


## Practice

Explain why normalizing all rows before splitting is leakage. Add a feature that directly contains the target and observe the misleading accuracy.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

The all-row transform uses information from the test distribution. A target-derived feature will be unavailable at prediction time; remove it even if offline accuracy improves.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
